In [1]:
# ===== Cell 1: path + imports =====
import sys, pathlib
ROOT = pathlib.Path.cwd().parent          # notebooks/ -> project root
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

from src.detection.preprocess import load_preprocessor, preprocess
from src.detection.detectors import load_all

In [2]:
# ===== Cell 2: load test data, saved preprocessor, saved detectors =====
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty",
]
df_test = pd.read_csv("../data/raw/KDDTest+.txt", header=None, names=columns)

scaler, cols = load_preprocessor("../artifacts")
dets = load_all("../artifacts")

X = preprocess(df_test, scaler, cols)                  # (22544, 122), scaled
y = (df_test["label"] != "normal").astype(int)         # 0 = normal, 1 = attack
print(X.shape, y.shape)

(22544, 122) (22544,)


In [3]:
# ===== Cell 3: Isolation Forest scores, A/B split =====
out = dets["isolation_forest"].predict(X)
print(out.keys())                                      # score, flagged, ratio
scores = np.asarray(out["score"])                      # higher = more anomalous
print(scores.shape, y.shape)                           # both (22544,)

scores_A, scores_B, y_A, y_B = train_test_split(
    scores, y, test_size=0.5, stratify=y, random_state=42
)
print(scores_A.shape, scores_B.shape)
print(y_A.mean(), y_B.mean())                          # both ~0.57

dict_keys(['score', 'flagged', 'ratio'])
(22544,) (22544,)
(11272,) (11272,)
0.5692867281760113 0.5691980127750177


In [4]:
# ===== Cell 4: logistic (Platt) calibrator =====
A = scores_A.reshape(-1, 1)                            # sklearn wants 2D features
B = scores_B.reshape(-1, 1)

cal = LogisticRegression()
cal.fit(A, y_A)                                        # learn w and b on half A
print(cal.coef_, cal.intercept_)                       # coef should be positive
p_B = cal.predict_proba(B)[:, 1]                       # P(attack) on half B
print("Brier logistic:", brier_score_loss(y_B, p_B))   # your run: 0.1046

[[20.90580336]] [-8.66988761]
Brier logistic: 0.10464067346266344


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights


In [5]:
# ===== Cell 5: isotonic calibrator =====
iso_cal = IsotonicRegression(out_of_bounds="clip")
iso_cal.fit(scores_A, y_A)                             # flat 1D scores, not A
p_B_iso = iso_cal.predict(scores_B)
print("Brier isotonic:", brier_score_loss(y_B, p_B_iso))   # your run: 0.0900

Brier isotonic: 0.08999750071173593


In [6]:
# ===== Cell 6: reliability table with counts =====
def reliability(y_true, p, n_bins=10):
    y_true = np.asarray(y_true)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(p, edges[1:-1])                  # bin number 0..n_bins-1
    rows = []
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        rows.append((b, p[m].mean(), y_true[m].mean(), int(m.sum())))
    return pd.DataFrame(rows, columns=["bin", "predicted", "actual", "n"])

print("LOGISTIC");  print(reliability(y_B, p_B).round(3))
print("ISOTONIC");  print(reliability(y_B, p_B_iso).round(3))

LOGISTIC
   bin  predicted  actual     n
0    1      0.137   0.030  3198
1    2      0.242   0.439   684
2    3      0.345   0.404   834
3    4      0.458   0.596   681
4    5      0.555   0.672   293
5    6      0.644   0.747   545
6    7      0.743   0.846   332
7    8      0.863   0.690   802
8    9      0.962   0.983  3903
ISOTONIC
   bin  predicted  actual     n
0    0      0.000   0.000  2987
1    2      0.278   0.407    27
2    3      0.390   0.439    41
3    4      0.429   0.428  1885
4    5      0.542   0.500    14
5    6      0.680   0.678  2040
6    8      0.855   0.939   147
7    9      0.977   0.981  4131


In [7]:
# ===== Cell 7: calibrate all three detectors (isotonic, capped to [0.01, 0.99]) =====
def calibrate(name):
    out = dets[name].predict(X)
    s = np.asarray(out["score"])
    sA, sB, yA, yB = train_test_split(s, y, test_size=0.5,
                                      stratify=y, random_state=42)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.01, y_max=0.99)
    iso.fit(sA, yA)
    p = iso.predict(sB)
    print(name, "Brier:", round(brier_score_loss(yB, p), 4))
    print(reliability(yB, p).round(3))
    return iso, p, yB

iso_if,  p_if,  y_B = calibrate("isolation_forest")
iso_mah, p_mah, _   = calibrate("mahalanobis")
iso_ae,  p_ae,  _   = calibrate("autoencoder")

isolation_forest Brier: 0.09
   bin  predicted  actual     n
0    0      0.010   0.000  2987
1    2      0.278   0.407    27
2    3      0.390   0.439    41
3    4      0.429   0.428  1885
4    5      0.542   0.500    14
5    6      0.680   0.678  2040
6    8      0.855   0.939   147
7    9      0.975   0.981  4131


/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: divide by zero encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: overflow encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: invalid value encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)


mahalanobis Brier: 0.0716
   bin  predicted  actual     n
0    0      0.010   0.001  2887
1    1      0.159   0.183  1294
2    2      0.240   0.206   131
3    3      0.393   0.358   109
4    4      0.417   0.538    13
5    5      0.522   0.529    17
6    6      0.633   0.631   977
7    7      0.782   0.891    46
8    8      0.879   0.875  2070
9    9      0.972   0.973  3728
autoencoder Brier: 0.0696
   bin  predicted  actual     n
0    0      0.010   0.001  3616
1    1      0.111   0.200    10
2    2      0.225   0.158    57
3    3      0.382   0.279    43
4    4      0.492   0.000     1
5    5      0.526   0.519  1210
6    6      0.617   0.474    57
7    7      0.716   0.688   625
8    8      0.881   0.886  2463
9    9      0.972   0.979  3190


In [8]:
d = X - dets["mahalanobis"].mu
alt = np.einsum("ij,jk,ik->i", d, dets["mahalanobis"].cov_inv, d)
orig = dets["mahalanobis"].predict(X)["score"]
print(np.nanmax(np.abs(alt - orig)))   # should be ~0

5.238689482212067e-10


/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: divide by zero encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: overflow encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: invalid value encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)


In [9]:
import joblib

# same rows as y_B, by index, so we can slice df_test later
idx_A, idx_B = train_test_split(np.arange(len(y)), test_size=0.5,
                                stratify=y, random_state=42)
half_B = pd.DataFrame({"idx": idx_B, "y": y.to_numpy()[idx_B],
                       "p_if": p_if, "p_mah": p_mah, "p_ae": p_ae})
assert (half_B["y"].to_numpy() == np.asarray(y_B)).all()   # confirms same rows

print(half_B[["p_if", "p_mah", "p_ae"]].corr().round(2))

# where Isolation Forest is unsure, does Mahalanobis separate attacks from normal?
zone = half_B[(half_B.p_if > 0.2) & (half_B.p_if < 0.9)]
print(len(zone), zone.y.mean())
print(zone.groupby(zone.p_mah > 0.5).y.agg(["mean", "size"]))

# save calibrators for the API / agent
joblib.dump(iso_if,  "../artifacts/calib_isolation_forest.joblib")
joblib.dump(iso_mah, "../artifacts/calib_mahalanobis.joblib")
joblib.dump(iso_ae,  "../artifacts/calib_autoencoder.joblib")

       p_if  p_mah  p_ae
p_if   1.00   0.90  0.91
p_mah  0.90   1.00  0.96
p_ae   0.91   0.96  1.00
4154 0.5690900337024555
           mean  size
p_mah                
False  0.215416  1453
True   0.759348  2701


['../artifacts/calib_autoencoder.joblib']

In [10]:
zone_df = df_test.iloc[zone["idx"]].copy()      # raw rows for the uncertain zone
zone_df["y"] = zone["y"].to_numpy()

tests = {
    "serror_rate > 0.5":   zone_df["serror_rate"] > 0.5,
    "logged_in == 1":      zone_df["logged_in"] == 1,
    "count > 100":         zone_df["count"] > 100,
    "same_srv_rate > 0.8": zone_df["same_srv_rate"] > 0.8,
}
for name, cond in tests.items():
    print(name)
    print(zone_df.groupby(cond)["y"].agg(["mean", "size"]))

serror_rate > 0.5
                mean  size
serror_rate               
False        0.56909  4154
logged_in == 1
               mean  size
logged_in                
False      0.546261  2367
True       0.599328  1787
count > 100
           mean  size
count                
False  0.615062  3572
True   0.286942   582
same_srv_rate > 0.8
                  mean  size
same_srv_rate               
False          0.64500   200
True           0.56525  3954


"exploration on half B, rules were re-derived on half A".

In [11]:
from sklearn.tree import DecisionTreeClassifier, export_text

# confirm idx_A lines up with y_A, as we did for half B
assert (y.to_numpy()[idx_A] == y_A.to_numpy()).all()

# Isolation Forest's uncertain zone, on half A
p_if_A = iso_if.predict(scores_A)
maskA = (p_if_A > 0.2) & (p_if_A < 0.9)
print("zone size on A:", maskA.sum(), " attack rate:", y_A.to_numpy()[maskA].mean())

rawA = df_test.iloc[idx_A][maskA]
XA = pd.get_dummies(rawA.drop(columns=["label", "difficulty"]))
yA_zone = y_A.to_numpy()[maskA]

tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=100, random_state=42)
tree.fit(XA, yA_zone)
print(export_text(tree, feature_names=list(XA.columns), show_weights=True))

zone size on A: 4189  attack rate: 0.5681546908570064
|--- duration <= 2.50
|   |--- service_domain_u <= 0.50
|   |   |--- weights: [1261.00, 1283.00] class: 1
|   |--- service_domain_u >  0.50
|   |   |--- weights: [460.00, 1.00] class: 0
|--- duration >  2.50
|   |--- dst_bytes <= 104.00
|   |   |--- weights: [9.00, 795.00] class: 1
|   |--- dst_bytes >  104.00
|   |   |--- weights: [79.00, 301.00] class: 1



In [13]:
# --- likelihood ratios from half A (with add-one smoothing) ---
leafA = tree.apply(XA)                         # which group each row falls into
tabA = pd.crosstab(leafA, yA_zone)
tabA.columns = ["normal", "attack"]
k = len(tabA)                                   # number of groups
tabA["LR"] = ((tabA["attack"] + 1) / (tabA["attack"].sum() + k)) / \
             ((tabA["normal"] + 1) / (tabA["normal"].sum() + k))
print(tabA.round(4))

# --- check the same rules on half B (one look, no re-tuning) ---
maskB = (p_if > 0.2) & (p_if < 0.9)             # p_if = half-B probabilities
rawB = df_test.iloc[idx_B][maskB]
XB = pd.get_dummies(rawB.drop(columns=["label", "difficulty"])) \
       .reindex(columns=XA.columns, fill_value=0)
yB_zone = y_B.to_numpy()[maskB]

leafB = tree.apply(XB)
tabB = pd.crosstab(leafB, yB_zone)
tabB.columns = ["normal", "attack"]
tabB["attack_rate"] = tabB["attack"] / (tabB["attack"] + tabB["normal"])
print(tabB.round(3))

       normal  attack       LR
row_0                         
2        1261    1283   0.7737
3         460       1   0.0033
5           9     795  60.5347
6          79     301   2.8708
       normal  attack  attack_rate
row_0                             
2        1295    1288        0.499
3         422       0        0.000
5           3     795        0.996
6          70     281        0.801


In [14]:
print(rawA[leafA == 5]["label"].value_counts().head(10))

label
guess_passwd    362
processtable    214
warezmaster     114
apache2          91
mscan             9
normal            9
rootkit           1
saint             1
named             1
satan             1
Name: count, dtype: int64


In [15]:
sc_mah = np.asarray(dets["mahalanobis"].predict(X)["score"])
p_mah_A = iso_mah.predict(sc_mah[idx_A])           # calibrated, half A
mahA = p_mah_A[maskA] > 0.5                         # Mahalanobis says attack?

g2 = leafA == 2                                     # the unresolved group
print(pd.crosstab(mahA[g2], yA_zone[g2]))           # rows: Mah says attack (F/T), cols: normal/attack

col_0    0     1
row_0           
False  676   254
True   585  1029


/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: divide by zero encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: overflow encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:61: RuntimeWarning: invalid value encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)


In [16]:
train_labels = set(pd.read_csv("../data/raw/KDDTrain+.txt",
                               header=None, names=columns)["label"])
grp5 = rawA[leafA == 5]["label"].value_counts()
print(grp5[~grp5.index.isin(train_labels)])      # group-5 labels never seen in training

label
processtable    214
apache2          91
mscan             9
saint             1
named             1
httptunnel        1
Name: count, dtype: int64


In [17]:
def lr_table(leaf, mah, y_zone):
    d = pd.DataFrame({"leaf": leaf, "mah": mah, "y": y_zone})
    rows = []
    for g, sub in d.groupby("leaf"):
        n_att, n_nor = (sub.y == 1).sum(), (sub.y == 0).sum()
        for flag in (False, True):
            s = sub[sub.mah == flag]
            a, nn = (s.y == 1).sum(), (s.y == 0).sum()
            lr = ((a + 1) / (n_att + 2)) / ((nn + 1) / (n_nor + 2))
            rate = a / (a + nn) if (a + nn) else np.nan
            rows.append((g, flag, nn, a, round(rate, 3), round(lr, 3)))
    return pd.DataFrame(rows, columns=["leaf", "mah_says_attack",
                                       "normal", "attack", "attack_rate", "LR"])

print("HALF A (where the ratios are estimated)")
print(lr_table(leafA, mahA, yA_zone))

mahB = p_mah[maskB] > 0.5                     # p_mah = half-B probabilities
print("\nHALF B (check only, no re-tuning)")
print(lr_table(leafB, mahB, yB_zone))

HALF A (where the ratios are estimated)
   leaf  mah_says_attack  normal  attack  attack_rate      LR
0     2            False     676     254        0.273   0.370
1     2             True     585    1029        0.638   1.728
2     3            False     441       0        0.000   0.348
3     3             True      19       1        0.050  15.400
4     5            False       5      25        0.833   0.060
5     5             True       4     770        0.995   2.128
6     6            False      30       4        0.118   0.043
7     6             True      49     297        0.858   1.593

HALF B (check only, no re-tuning)
   leaf  mah_says_attack  normal  attack  attack_rate      LR
0     2            False     713     288        0.288   0.407
1     2             True     582    1000        0.632   1.726
2     3            False     402       0        0.000   0.526
3     3             True      20       0        0.000  10.095
4     5            False       3      22        0.880   0

In [18]:
sc_ae  = np.asarray(dets["autoencoder"].predict(X)["score"])
p_ae_A = iso_ae.predict(sc_ae[idx_A])

avg_A = (p_mah_A + p_ae_A) / 2
avg_B = (p_mah   + p_ae)   / 2

comb = IsotonicRegression(out_of_bounds="clip", y_min=0.01, y_max=0.99)
comb.fit(avg_A, y_A)
pc_A = comb.predict(avg_A)
pc_B = comb.predict(avg_B)
print("Brier combined prior, half B:", brier_score_loss(y_B, pc_B))   # about 0.0650

Brier combined prior, half B: 0.06504559545730994


In [19]:
maskA_c = (pc_A > 0.2) & (pc_A < 0.9)
maskB_c = (pc_B > 0.2) & (pc_B < 0.9)
print("zone sizes:", maskA_c.sum(), maskB_c.sum())                  # 3968, 3931
print("attack rate:", y_A.to_numpy()[maskA_c].mean(),
      y_B.to_numpy()[maskB_c].mean())                               # about 0.72, 0.72

rawA_c = df_test.iloc[idx_A][maskA_c]
rawB_c = df_test.iloc[idx_B][maskB_c]
XA_c = pd.get_dummies(rawA_c.drop(columns=["label", "difficulty"]))
XB_c = pd.get_dummies(rawB_c.drop(columns=["label", "difficulty"])) \
         .reindex(columns=XA_c.columns, fill_value=0)
yA_zone_c = y_A.to_numpy()[maskA_c]
yB_zone_c = y_B.to_numpy()[maskB_c]

tree_c = DecisionTreeClassifier(max_depth=2, min_samples_leaf=100, random_state=42)
tree_c.fit(XA_c, yA_zone_c)
print(export_text(tree_c, feature_names=list(XA_c.columns), show_weights=True))

zone sizes: 3968 3931
attack rate: 0.7245463709677419 0.7189010429916052
|--- dst_host_same_src_port_rate <= 0.00
|   |--- dst_host_rerror_rate <= 0.00
|   |   |--- weights: [100.00, 171.00] class: 1
|   |--- dst_host_rerror_rate >  0.00
|   |   |--- weights: [84.00, 1758.00] class: 1
|--- dst_host_same_src_port_rate >  0.00
|   |--- dst_bytes <= 2.50
|   |   |--- weights: [304.00, 706.00] class: 1
|   |--- dst_bytes >  2.50
|   |   |--- weights: [605.00, 240.00] class: 0



In [20]:
leafA_c = tree_c.apply(XA_c)
tabA_c = pd.crosstab(leafA_c, yA_zone_c)
tabA_c.columns = ["normal", "attack"]
k = len(tabA_c)
tabA_c["LR"] = ((tabA_c["attack"] + 1) / (tabA_c["attack"].sum() + k)) / \
               ((tabA_c["normal"] + 1) / (tabA_c["normal"].sum() + k))
print(tabA_c.round(4))

leafB_c = tree_c.apply(XB_c)
tabB_c = pd.crosstab(leafB_c, yB_zone_c)
tabB_c.columns = ["normal", "attack"]
tabB_c["attack_rate"] = tabB_c["attack"] / (tabB_c["attack"] + tabB_c["normal"])
print(tabB_c.round(3))

       normal  attack      LR
row_0                        
2         100     171  0.6489
3          84    1758  7.8852
5         304     706  0.8833
6         605     240  0.1515
       normal  attack  attack_rate
row_0                             
2         101     142        0.584
3          90    1771        0.952
5         307     655        0.681
6         607     258        0.298


In [21]:
ifA = p_if_A[maskA_c] > 0.5 # for the isolation forrest 
ifB = p_if[maskB_c]  > 0.5

def if_table(leaf, flag, y_zone):
    rows = []
    for g in sorted(set(leaf)):
        in_g = leaf == g
        n_att = (y_zone[in_g] == 1).sum()
        n_nor = (y_zone[in_g] == 0).sum()
        for f in (False, True):
            s = in_g & (flag == f)
            a  = (y_zone[s] == 1).sum()
            n0 = (y_zone[s] == 0).sum()
            lr = ((a + 1) / (n_att + 2)) / ((n0 + 1) / (n_nor + 2))
            rows.append((g, f, n0, a, round(a / max(a + n0, 1), 3), round(lr, 3)))
    return pd.DataFrame(rows, columns=["leaf", "if_says_attack",
                                       "normal", "attack", "attack_rate", "LR"])

print("HALF A"); print(if_table(leafA_c, ifA, yA_zone_c))
print("HALF B"); print(if_table(leafB_c, ifB, yB_zone_c))

HALF A
   leaf  if_says_attack  normal  attack  attack_rate     LR
0     2           False      27      13        0.325  0.295
1     2            True      73     158        0.684  1.267
2     3           False      26     249        0.905  0.452
3     3            True      58    1509        0.963  1.251
4     5           False     137      52        0.275  0.166
5     5            True     167     654        0.797  1.685
6     6           False     212     104        0.329  1.236
7     6            True     393     136        0.257  0.872
HALF B
   leaf  if_says_attack  normal  attack  attack_rate     LR
0     2           False      34      15        0.306  0.327
1     2            True      67     127        0.655  1.346
2     3           False      36     288        0.889  0.405
3     3            True      54    1483        0.965  1.400
4     5           False     150      50        0.250  0.159
5     5            True     157     605        0.794  1.804
6     6           False   

In [22]:
def to_odds(p):  return p / (1 - p)
def to_prob(o):  return o / (1 + o)

lr_group = {int(g): v for g, v in tabA_c["LR"].items()}
tIF = if_table(leafA_c, ifA, yA_zone_c)
lr_if_all = {(int(r.leaf), bool(r.if_says_attack)): r.LR for r in tIF.itertuples()}
lr_if_g5  = {k: v for k, v in lr_if_all.items() if k[0] == 5}
base = yA_zone_c.mean()

def agent(leaf, if_flag, lr_if):
    o = to_odds(base) * lr_group[int(leaf)]
    key = (int(leaf), bool(if_flag))
    if key in lr_if:
        o *= lr_if[key]
    return float(np.clip(to_prob(o), 0.001, 0.999))

versions = {"tree only": {},
            "tree + IF (all groups)": lr_if_all,
            "tree + IF (group 5 only)": lr_if_g5}
post = {name: np.array([agent(g, f, d) for g, f in zip(leafB_c, ifB)])
        for name, d in versions.items()}

print(f"{'combined prior (no agent)':26s} zone Brier",
      round(brier_score_loss(yB_zone_c, pc_B[maskB_c]), 4))
for name, v in post.items():
    print(f"{name:26s} zone Brier", round(brier_score_loss(yB_zone_c, v), 4))

combined prior (no agent)  zone Brier 0.1677
tree only                  zone Brier 0.1363
tree + IF (all groups)     zone Brier 0.1229
tree + IF (group 5 only)   zone Brier 0.1243


In [23]:
from sklearn.metrics import precision_score, recall_score, f1_score

yb = y_B.to_numpy()

def show(name, p):
    pred = (p > 0.5).astype(int)
    print(f"{name:26s} {brier_score_loss(yb, p):.4f}   "
          f"{precision_score(yb, pred):.3f}   {recall_score(yb, pred):.3f}   {f1_score(yb, pred):.3f}")

print(f"{'method':26s} Brier    precision recall  F1")
show("raw Mahalanobis", p_mah)
show("raw autoencoder", p_ae)
show("combined prior", pc_B)
for name, v in post.items():
    p_full = pc_B.copy()
    p_full[maskB_c] = v
    show(name, p_full)

print("\ncalibration, group-5-only agent, half B zone")
print(reliability(yB_zone_c, post["tree + IF (group 5 only)"]).round(3))

method                     Brier    precision recall  F1
raw Mahalanobis            0.0716   0.893   0.951   0.921
raw autoencoder            0.0696   0.847   0.996   0.916
combined prior             0.0650   0.896   0.954   0.924
tree only                  0.0541   0.915   0.959   0.937
tree + IF (all groups)     0.0494   0.941   0.949   0.945
tree + IF (group 5 only)   0.0499   0.936   0.951   0.943

calibration, group-5-only agent, half B zone
   bin  predicted  actual     n
0    2      0.284   0.289  1065
1    6      0.631   0.584   243
2    7      0.797   0.794   762
3    9      0.954   0.952  1861


In [24]:
GROUP_TEXT = {
    2: "no repeated source port and no rejected connections to the destination host",
    3: "no repeated source port, with rejected connections to the destination host",
    5: "a repeated source port and almost no data returned",
    6: "a repeated source port and some data returned",
}

def agent_trace(leaf, if_flag, lr_if):
    leaf = int(leaf)
    o = to_odds(base)
    steps = [("start", 1.0, base)]
    o *= lr_group[leaf]
    steps.append((GROUP_TEXT[leaf], lr_group[leaf], to_prob(o)))
    key = (leaf, bool(if_flag))
    if key in lr_if:
        o *= lr_if[key]
        label = "Isolation Forest says attack" if if_flag else "Isolation Forest says normal"
        steps.append((label, lr_if[key], to_prob(o)))
    return steps

def verdict(p):
    if p >= 0.9: return "ESCALATE"
    if p >= 0.5: return "HUMAN REVIEW"
    return "LOG (low priority)"

def investigate(leaf, if_flag):
    steps = agent_trace(leaf, if_flag, lr_if_g5)
    p = float(np.clip(steps[-1][2], 0.001, 0.999))
    lines = [f"Start: {steps[0][2]:.0%} attack in the uncertain zone"]
    lines += [f"{label} (x{ratio:.2f}) -> {prob:.0%}" for label, ratio, prob in steps[1:]]
    return {"probability": p, "action": verdict(p), "reason": " | ".join(lines)}

chk = np.array([investigate(g, f)["probability"] for g, f in zip(leafB_c, ifB)])
print("max difference vs agent():", np.abs(chk - post["tree + IF (group 5 only)"]).max())

seen = set()
for g, f in zip(leafB_c, ifB):
    if int(g) not in seen:
        seen.add(int(g)); print(int(g), investigate(g, f))

max difference vs agent(): 0.0
3 {'probability': 0.9540039427290655, 'action': 'ESCALATE', 'reason': 'Start: 72% attack in the uncertain zone | no repeated source port, with rejected connections to the destination host (x7.89) -> 95%'}
5 {'probability': 0.7965301407180215, 'action': 'HUMAN REVIEW', 'reason': 'Start: 72% attack in the uncertain zone | a repeated source port and almost no data returned (x0.88) -> 70% | Isolation Forest says attack (x1.69) -> 80%'}
2 {'probability': 0.6305638732841621, 'action': 'HUMAN REVIEW', 'reason': 'Start: 72% attack in the uncertain zone | no repeated source port and no rejected connections to the destination host (x0.65) -> 63%'}
6 {'probability': 0.28499448681073447, 'action': 'LOG (low priority)', 'reason': 'Start: 72% attack in the uncertain zone | a repeated source port and some data returned (x0.15) -> 28%'}


In [25]:
p_full = pc_B.copy()
p_full[maskB_c] = post["tree + IF (group 5 only)"]
tri = pd.DataFrame({"action": [verdict(p) for p in p_full], "y": yb})
print(tri.groupby("action")["y"].agg(rows="size", attack_rate="mean").round(3))
print("attacks caught by ESCALATE + REVIEW:",
      round(tri[tri.action != "LOG (low priority)"].y.sum() / tri.y.sum(), 3))

                    rows  attack_rate
action                               
ESCALATE            5516        0.971
HUMAN REVIEW        1005        0.743
LOG (low priority)  4751        0.066
attacks caught by ESCALATE + REVIEW: 0.951


In [26]:
import json, joblib

joblib.dump(tree_c, "../artifacts/agent_tree.joblib")
joblib.dump(comb,   "../artifacts/agent_calibrator.joblib")

params = {
    "base_rate": float(base),
    "lr_group": {str(g): float(v) for g, v in lr_group.items()},
    "lr_if": {f"{g},{int(flag)}": float(v) for (g, flag), v in lr_if_g5.items()},
    "tree_columns": list(XA_c.columns),
    "zone_low": 0.2, "zone_high": 0.9,
    "if_threshold": 0.5,
    "escalate_at": 0.9, "review_at": 0.5,
    "group_text": {str(g): t for g, t in GROUP_TEXT.items()},
}
json.dump(params, open("../artifacts/agent_params.json", "w"), indent=2)
print(sorted(params))

['base_rate', 'escalate_at', 'group_text', 'if_threshold', 'lr_group', 'lr_if', 'review_at', 'tree_columns', 'zone_high', 'zone_low']
